## Sheppard_2020, cellxgene counts, batch dataset, kl 2

In [ ]:
import scanpy as sc
import sys
import os
import pandas as pd


sys.path.append("../../../")
sys.path.append("../../utils/HLCA")

import mapqc
import data_cleanup

In [ ]:
sc.set_figure_params(figsize=(5, 5))

In [ ]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

In [ ]:
study = "Sheppard_2020"
batch_variable = "dataset"
kl = "2.0"
k = 500
max_k = 5000
n_nhoods = 500
testrun = False
use_old_counts = False
distance_type = "edistance"

In [ ]:
if testrun:
    suffix = "_test"
    suffix_2 = ""
else:
    suffix = ""
    if use_old_counts == True:
        suffix_2 = "original_counts_"
    else:
        suffix_2 = "cellxgene_counts_"

path_emb = f"/lustre/groups/ml01/workspace/lisa.sikkema/mapqc/data/HLCA/HLCA_and_{study}_{suffix_2}batch_{batch_variable}_kl{kl}_latent{suffix}.h5ad"
path_counts = f"/lustre/groups/ml01/workspace/lisa.sikkema/mapqc/data/HLCA/HLCA_and_{study}_counts{suffix}.h5ad"
path_results_dir = f"/lustre/groups/ml01/code/lisa.sikkema/mapqc/results/HLCA/{study}/{suffix_2}batch_{batch_variable}_kl{kl}{suffix}/mapqc"

check if results dir exists, if it's parent directory exists then create the final ("mapqc") directory

In [ ]:
# check if parent directory exists:
parent_dir = os.path.dirname(path_results_dir)
if not os.path.isdir(parent_dir):
    raise ValueError(
        f"Parent directory of the specified results directory ({parent_dir}) does not exist."
    )
else:
    if not os.path.isdir(path_results_dir):
        # create directory:
        print(f"Creating directory {path_results_dir}...")
        os.makedirs(path_results_dir)

In [ ]:
ref_q_key = "r_or_q"
ref_cat = "r"
query_cat = "q"

In [ ]:
adata_emb = sc.read_h5ad(path_emb)
adata_counts = sc.read_h5ad(path_counts)

In [ ]:
adata = data_cleanup.add_mapping_info_to_counts_adata(adata_counts, adata_emb)

In [ ]:
# results_healthy = mapqc.MapqcResults(
#     adata=adata,
#     ref_q_key="r_or_q",
#     ref_cat="r",
#     query_cat="q",
#     condition_key="status",
#     control_cats="control",
#     control_or_case="control",
#     case_cats="IPF",
#     emb_key="X_scanvi_emb",
#     dataset_key="study",
#     sample_key="sample",
#     sampling_group_cat="leiden",
#     distance_type=distance_type,
#     seed=0,
#     n_cores=2,
#     k=k,
#     max_k=max_k,
#     n_nhoods=n_nhoods,
# )
results_healthy.run_mapqc()
# results_healthy.run_mapqc()
# results_disease = mapqc.MapqcResults(
#     adata=adata,
#     ref_q_key="r_or_q",
#     ref_cat="r",
#     query_cat="q",
#     condition_key="status",
#     control_cats="control",
#     control_or_case="case",
#     case_cats="IPF",
#     emb_key="X_scanvi_emb",
#     dataset_key="study",
#     sample_key="sample",
#     sampling_group_cat="leiden",
#     distance_type=distance_type,
#     seed=0,
#     n_cores=2,
#     k=k,
#     max_k=max_k,
#     n_nhoods=n_nhoods,
# )

# results_disease.run_mapqc()

store results:

In [ ]:
if distance_type == "pairwise_euclidean":
    results_healthy.save(os.path.join(path_results_dir, "control.pkl"))
    results_disease.save(os.path.join(path_results_dir, "case.pkl"))
elif distance_type in ["centroid_euclidean", "edistance"]:
    dist_suffix = distance_type.split("_")[0]
    results_healthy.save(
        os.path.join(path_results_dir, f"control_{dist_suffix}_k{k}.pkl")
    )
    results_disease.save(os.path.join(path_results_dir, f"case_{dist_suffix}_k{k}.pkl"))
else:
    raise ValueError("Unknown distance type")

load results if starting from here:

In [ ]:
if distance_type == "pairwise_euclidean":
    results_healthy = mapqc.load(os.path.join(path_results_dir, "control.pkl"))
    results_disease = mapqc.load(os.path.join(path_results_dir, "case.pkl"))
elif distance_type in ["centroid_euclidean", "edistance"]:
    dist_suffix = distance_type.split("_")[0]
    results_healthy = mapqc.load(
        os.path.join(path_results_dir, f"control_{dist_suffix}_k{k}.pkl")
    )
    results_disease = mapqc.load(
        os.path.join(path_results_dir, f"case_{dist_suffix}_k{k}.pkl")
    )
else:
    raise ValueError("Unknown distance type")

In [ ]:
results_healthy.result_stats

In [ ]:
results_disease.result_stats

In [ ]:
cmap = {"r control": "grey", "q control": "dodgerblue", "q IPF": "midnightblue"}

In [ ]:
# create variable splitting data in ref control, query control
adata.obs["r_q_status"] = [
    f"{r_q} {stat}" for r_q, stat in zip(adata.obs.r_or_q, adata.obs.status)
]

In [ ]:
sc.pl.umap(
    adata,
    color="r_q_status",
    palette=cmap,
    frameon=False,
)

In [ ]:
adata.obs["donor_id_q"] = None
adata.obs.loc[adata.obs.r_or_q == "q", "donor_id_q"] = adata.obs.loc[
    adata.obs.r_or_q == "q", "donor_id"
].tolist()

In [ ]:
sc.pl.umap(adata, color=["donor_id_q", "study"], frameon=False)

### Results healthy

In [ ]:
mapqc.pl.umap_mapqc_scores(results_healthy, color_by="mean_pval_corr", size=0.5)

In [ ]:
mapqc.pl.umap_mapqc_scores(
    results_healthy, color_by="mean_pval_corr_significant", size=0.5
)

In [ ]:
mapqc.pl.umap_mapqc_scores(results_healthy, color_by="mapqc_score>2", size=0.5)

In [ ]:
mapqc.pl.umap_mapqc_scores(results_healthy, size=0.5)

In [ ]:
mapqc.pl.umap_filtering_info(results_healthy)

In [ ]:
mapqc.pl.boxplot_sample_scores_per_nhood(
    results_healthy, label_x_ticks_by="ann_level_4_pred"
)

In [ ]:
mapqc.pl.boxplot_mapqc_scores_per_category(results_healthy, box_category="sample")

In [ ]:
mapqc.pl.boxplot_mapqc_scores_per_category(
    results_healthy, box_category="ann_level_2_pred", color_category="donor_id"
)

In [ ]:
mapqc.pl.boxplot_mapqc_scores_per_category(
    results_healthy, box_category="ann_level_2_pred", color_category="donor_id"
)

In [ ]:
mapqc.pl.boxplot_mapqc_scores_per_category(
    results_healthy, box_category="ann_level_3_pred", color_category="donor_id"
)

In [ ]:
custom_palette = {"control": "dodgerblue", "IPF": "midnightblue"}

In [ ]:
mapqc.pl.grouped_boxplot_mapqc_scores_per_category(
    results_healthy,
    results_disease,
    box_category="ann_level_2_pred",
    group_category="status",
    palette=custom_palette,
    min_cells_per_box=50,
)

In [ ]:
mapqc.pl.grouped_boxplot_mapqc_scores_per_category(
    results_healthy,
    results_disease,
    box_category="ann_level_3_pred",
    group_category="status",
    palette=custom_palette,
    min_cells_per_box=50,
)

In [ ]:
mapqc.pl.grouped_boxplot_mapqc_scores_per_category(
    results_healthy,
    results_disease,
    box_category="ann_level_4_pred",
    group_category="status",
    palette=custom_palette,
    min_cells_per_box=50,
)

### Results disease

In [ ]:
mapqc.pl.umap_mapqc_scores(results_disease, size=1)

In [ ]:
mapqc.pl.umap_filtering_info(results_disease)

In [ ]:
mapqc.pl.boxplot_sample_scores_per_nhood(results_disease)

In [ ]:
mapqc.pl.boxplot_mapqc_scores_per_category(results_disease, box_category="sample")

In [ ]:
mapqc.pl.boxplot_mapqc_scores_per_category(
    results_disease, box_category="ann_level_2_pred", color_category="donor_id"
)

In [ ]:
mapqc.pl.boxplot_mapqc_scores_per_category(
    results_disease, box_category="ann_level_3_pred", color_category="donor_id"
)

In [ ]:
mapqc.pl.boxplot_mapqc_scores_per_category(
    results_disease, box_category="ann_level_4_pred", color_category="donor_id"
)